# Deepfake Audio Detection System
### *Vishing (Voice Phishing) Defense — Model Training Notebook*

---

This notebook will:
1. Install all required libraries
2. Let you upload your audio dataset (real + fake voices)
3. Extract audio features using librosa
4. Train a Random Forest classifier
5. Evaluate the model (accuracy, confusion matrix, etc.)
6. Download `voice_model.pkl` — ready for the Streamlit app
7. Launch the Streamlit web app via **ngrok** (no password needed)

---

### Before You Start
Prepare two folders of audio files on your computer:
- `real/` — real human voice recordings (`.wav`, `.mp3`, `.flac`, or `.ogg`)
- `fake/` — AI-generated / deepfake voice recordings

> **Tip:** Minimum 20 files per folder. More files = better accuracy.


---

### Notebook Flow

| Step | Description |
|------|-------------|
| 1 | Install libraries |
| 2 | Import packages |
| 3 | Upload dataset |
| 4 | Validate dataset |
| 5 | Feature extraction |
| 6 | Train model |
| 7 | Evaluate model |
| 8 | Preview audio visuals (optional) |
| 9 | Save and download model |
| 10 | Test single prediction (optional) |
| 11 | Run Streamlit app via ngrok |

---
## Step 1 — Install Libraries
Run this cell first. It installs all required Python packages.
Wait for the done message before moving to Step 2.

The **!pip install** command is used to run pip commands directly in the Colab environment. It's installing **librosa**, **scikit-learn**,** matplotlib**, **seaborn**, **numpy**, **pandas**, and **tqdm**. The **--quiet** flag suppresses most of the installation output, making it less verbose. After the installations are complete, it prints "All libraries installed successfully!" to confirm the process.

In [ ]:
!pip install librosa scikit-learn matplotlib seaborn numpy pandas tqdm --quiet

print("All libraries installed successfully!")

---
## Step 2 — Import Libraries

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import librosa
import librosa.display

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
import zipfile
import shutil
from tqdm import tqdm

print("All imports successful!")

---
## Step 3 — Upload Your Dataset

**Upload a single ZIP file containing both folders:**
```
dataset.zip
  real/
    voice1.wav
    voice2.wav
  fake/
    deepfake1.wav
    deepfake2.wav
```


In [ ]:
# Upload a ZIP file


os.makedirs("dataset/real", exist_ok=True)
os.makedirs("dataset/fake", exist_ok=True)

print("Please upload your dataset ZIP file...")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith(".zip"):
        print(f"Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall(".")
        print("Extraction complete!")
    else:
        print(f"{filename} is not a ZIP file. Please use Option B instead.")

real_files = [f for f in os.listdir("dataset/real") if f.lower().endswith((".wav",".mp3",".flac",".ogg"))]
fake_files = [f for f in os.listdir("dataset/fake") if f.lower().endswith((".wav",".mp3",".flac",".ogg"))]

print(f"\nDataset Summary:")
print(f"  real/ --> {len(real_files)} audio file(s)")
print(f"  fake/ --> {len(fake_files)} audio file(s)")

---
## Step 4 — Validate Dataset

In [ ]:
AUDIO_EXTENSIONS = (".wav", ".mp3", ".flac", ".ogg")

real_files = [f for f in os.listdir("dataset/real") if f.lower().endswith(AUDIO_EXTENSIONS)]
fake_files = [f for f in os.listdir("dataset/fake") if f.lower().endswith(AUDIO_EXTENSIONS)]

print("=" * 50)
print("  DATASET VALIDATION")
print("=" * 50)
print(f"  real/  -->  {len(real_files):>4} audio file(s)")
print(f"  fake/  -->  {len(fake_files):>4} audio file(s)")
print("=" * 50)

errors = []
if len(real_files) == 0:
    errors.append("dataset/real/ is empty. Please go back to Step 3.")
if len(fake_files) == 0:
    errors.append("dataset/fake/ is empty. Please go back to Step 3.")
if len(real_files) < 10 and len(real_files) > 0:
    print(f"  WARNING: Only {len(real_files)} real files. Recommend 20+ for good accuracy.")
if len(fake_files) < 10 and len(fake_files) > 0:
    print(f"  WARNING: Only {len(fake_files)} fake files. Recommend 20+ for good accuracy.")

if errors:
    for e in errors:
        print(f"  ERROR: {e}")
    raise ValueError("Dataset missing. Please upload audio files in Step 3 and re-run.")
else:
    print("  Dataset looks good — ready to train!")

---
## Step 5 — Feature Extraction

Converts each audio file into a 154-number feature vector using librosa.

| Feature | What it captures |
|---|---|
| MFCC (40 coefficients) | Voice fingerprint — most powerful for detecting fake speech |
| Chroma | 12 pitch class energies |
| Spectral Contrast | Peaks vs valleys per frequency band |
| Zero Crossing Rate | How often signal crosses zero — higher in synthetic speech |
| RMS Energy | Volume envelope over time |

In [ ]:
def extract_features(file_path, n_mfcc=40):
    """
    Load one audio file and return a 154-number feature vector.
    Returns None if the file cannot be processed.
    """
    try:
        # Load audio: 22050 Hz sample rate, max 5 seconds
        audio, sample_rate = librosa.load(file_path, sr=22050, duration=5.0)

        # Extract each feature type
        mfccs    = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
        chroma   = librosa.feature.chroma_stft(y=audio, sr=sample_rate)
        contrast = librosa.feature.spectral_contrast(y=audio, sr=sample_rate)
        zcr      = librosa.feature.zero_crossing_rate(y=audio)
        rms      = librosa.feature.rms(y=audio)

        # Summarise each matrix: mean, std, max per coefficient
        def summarise(feat):
            return np.concatenate([
                np.mean(feat, axis=1),
                np.std(feat,  axis=1),
                np.max(feat,  axis=1)
            ])

        # Combine into one flat 154-number vector
        return np.concatenate([
            summarise(mfccs),
            summarise(chroma),
            summarise(contrast),
            [np.mean(zcr), np.std(zcr)],
            [np.mean(rms), np.std(rms)]
        ])

    except Exception as e:
        print(f"  Skipping {os.path.basename(file_path)}: {e}")
        return None


def load_dataset(dataset_path="dataset"):
    """Load all audio files. Returns feature matrix X and labels y."""
    X, y = [], []
    classes = {"real": 0, "fake": 1}   # 0 = real, 1 = deepfake

    for folder_name, label in classes.items():
        folder_path = os.path.join(dataset_path, folder_name)
        audio_files = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith((".wav", ".mp3", ".flac", ".ogg"))
        ]
        print(f"\n  Extracting from '{folder_name}/' ({len(audio_files)} files)...")
        for filename in tqdm(audio_files, desc=f"  {folder_name}"):
            features = extract_features(os.path.join(folder_path, filename))
            if features is not None:
                X.append(features)
                y.append(label)

    return np.array(X), np.array(y)


print("Extracting audio features — please wait...")
X, y = load_dataset("dataset")

print(f"\nFeature extraction complete!")
print(f"  Total samples  : {len(X)}")
print(f"  Real voices    : {(y == 0).sum()}")
print(f"  Deepfakes      : {(y == 1).sum()}")
print(f"  Feature vector : {X.shape[1]} dimensions per file")

---
## Step 6 — Train the Model

In [ ]:
# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"  Training samples : {len(X_train)}")
print(f"  Testing  samples : {len(X_test)}")

# Normalise features to mean=0, std=1
# fit() on training data only — then transform both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train the Random Forest
print("\nTraining Random Forest (200 trees)... please wait...")

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

print("Training complete!")

---
## Step 7 — Evaluate the Model

In [ ]:
y_pred = model.predict(X_test_scaled)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)

print("=" * 45)
print("  MODEL EVALUATION RESULTS")
print("=" * 45)
print(f"  Accuracy  : {acc  * 100:.2f} %")
print(f"  Precision : {prec * 100:.2f} %")
print(f"  Recall    : {rec  * 100:.2f} %")
print(f"  F1 Score  : {f1   * 100:.2f} %")
print("=" * 45)
print()
print(classification_report(y_test, y_pred, target_names=["Real Voice", "Deepfake"]))

In [ ]:
# Plot confusion matrix and metrics chart
cm   = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0d1117")

ax1 = axes[0]
ax1.set_facecolor("#161b22")
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=["Real Voice", "Deepfake"],
            yticklabels=["Real Voice", "Deepfake"],
            linewidths=2, linecolor="#0d1117", ax=ax1,
            annot_kws={"size": 16, "weight": "bold", "color": "white"})
ax1.set_title("Confusion Matrix", color="white", fontsize=14, pad=12)
ax1.set_xlabel("Predicted", color="#8b949e")
ax1.set_ylabel("Actual",    color="#8b949e")
ax1.tick_params(colors="white")

ax2 = axes[1]
ax2.set_facecolor("#161b22")
names  = ["Accuracy", "Precision", "Recall", "F1 Score"]
values = [acc, prec, rec, f1]
colors = ["#58a6ff", "#3fb950", "#f78166", "#d2a8ff"]
bars   = ax2.bar(names, values, color=colors, edgecolor="#0d1117", linewidth=1.5, width=0.55)
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
             f"{val*100:.1f}%", ha="center", va="bottom",
             color="white", fontsize=12, fontweight="bold")
ax2.set_ylim(0, 1.15)
ax2.set_title("Performance Metrics", color="white", fontsize=14, pad=12)
ax2.tick_params(colors="white")
ax2.set_ylabel("Score", color="#8b949e")
for spine in ax2.spines.values(): spine.set_color("#30363d")

fig.suptitle("Deepfake Audio Detection — Model Evaluation",
             color="white", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("model_evaluation.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Chart saved as model_evaluation.png")

---
## Step 8 — Preview Audio Visuals (Optional)
Shows the waveform for one real and one fake sample.

This code defines a function to visualize audio waveforms and then uses it to display one real and one deepfake audio sample from your dataset.


In [ ]:
def plot_audio_preview(file_path, label_name, color):
    audio, sr = librosa.load(file_path, sr=22050, duration=5.0)
    fig, ax = plt.subplots(figsize=(10, 3))
    fig.patch.set_facecolor("#0d1117")
    ax.set_facecolor("#161b22")
    times = np.linspace(0, len(audio)/sr, len(audio))
    ax.plot(times, audio, color=color, linewidth=0.7)
    ax.fill_between(times, audio, alpha=0.2, color=color)
    ax.set_title(f"Waveform — {label_name}", color="white", fontsize=11)
    ax.tick_params(colors="#8b949e")
    for spine in ax.spines.values(): spine.set_color("#30363d")
    plt.tight_layout()
    plt.show()

real_sample = next((os.path.join("dataset/real", f) for f in os.listdir("dataset/real")
                    if f.lower().endswith((".wav",".mp3",".flac",".ogg"))), None)
fake_sample = next((os.path.join("dataset/fake", f) for f in os.listdir("dataset/fake")
                    if f.lower().endswith((".wav",".mp3",".flac",".ogg"))), None)

if real_sample: plot_audio_preview(real_sample, "Real Voice",     "#3fb950")
if fake_sample: plot_audio_preview(fake_sample, "Deepfake Voice", "#f85149")


---
## Step 8b — MFCC Visualization (Optional)
Displays the **Mel-Frequency Cepstral Coefficients (MFCC)** heatmap for one real and one fake sample.

MFCCs represent the short-term power spectrum of audio and are the most important feature for detecting deepfakes. Each row is one coefficient (0–39), and each column is a time frame. AI-generated voices often show unnaturally uniform or repeating patterns across the heatmap compared to real speech.


In [ ]:
def plot_mfcc_preview(file_path, label_name, is_fake):
    """
    Load one audio file and display its MFCC heatmap.
    Green colour map for real voice, red for deepfake.
    """
    audio, sr = librosa.load(file_path, sr=22050, duration=5.0)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    cmap  = "RdPu" if is_fake else "GnBu"

    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor("#0d1117")
    ax.set_facecolor("#161b22")

    img = librosa.display.specshow(
        mfccs, sr=sr, x_axis="time", ax=ax, cmap=cmap
    )
    fig.colorbar(img, ax=ax, format="%+.1f", label="Coefficient value")

    color = "#f85149" if is_fake else "#3fb950"
    ax.set_title(
        f"MFCC Heatmap — {label_name}",
        color=color, fontsize=12, pad=10
    )
    ax.set_xlabel("Time (seconds)",        color="#8b949e", fontsize=9)
    ax.set_ylabel("MFCC Coefficient",      color="#8b949e", fontsize=9)
    ax.tick_params(colors="#8b949e")
    for spine in ax.spines.values():
        spine.set_color("#30363d")

    plt.tight_layout()
    plt.show()
    print(f"  Rows  = 40 MFCC coefficients  |  Columns = time frames")
    print(f"  Brighter cells = higher coefficient energy at that moment\n")


# Plot MFCC for one real and one fake sample
if real_sample:
    plot_mfcc_preview(real_sample, "Real Voice",     is_fake=False)
if fake_sample:
    plot_mfcc_preview(fake_sample, "Deepfake Voice", is_fake=True)


---
## Step 9 — Save and Download the Model

- **Running Streamlit in Colab (Step 11)?** No need to download — the file is already in the right place.
- **Running Streamlit locally in VS Code?** Download it and place it in your project folder.

In [ ]:
# Save model + scaler together into one file
MODEL_PATH = "voice_model.pkl"

payload = {"model": model, "scaler": scaler}
with open(MODEL_PATH, "wb") as f:
    pickle.dump(payload, f)

size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f"Model saved as '{MODEL_PATH}'  ({size_kb:.1f} KB)")
print()
print("Running Streamlit in Colab (Step 11)?")
print("  --> No download needed. voice_model.pkl is already here.")
print()
print("Running Streamlit locally in VS Code?")
print("  --> Run the download cell below.")

In [ ]:
# OPTIONAL download — only needed for running locally in VS Code
# Skip this cell if you are running Streamlit inside Colab (Step 11)

files.download("voice_model.pkl")
print("Download started! Move voice_model.pkl into your VS Code project folder.")

---
## Step 10 — Test a Single Prediction (Optional)
Upload any one audio file and test the model directly here in Colab.
Good for a quick check before launching the full web app.
```
This code allows you to upload a single audio file and immediately get a prediction from your trained model.

First, it prompts you to upload an audio file.

Once uploaded, it extracts the audio features, scales them using the previously fitted scaler, and then uses your Random Forest model to predict if it's a real or deepfake voice.

Finally, it displays the prediction, its confidence, and the probabilities for both 'real' and 'fake' categories.
```

In [ ]:
print("Upload a single audio file to test...")
test_upload = files.upload()

for test_filename in test_upload.keys():
    print(f"\nAnalysing: {test_filename}")
    features = extract_features(test_filename)

    if features is None:
        print("Could not process the file. Make sure it is a valid audio file.")
    else:
        features_scaled = scaler.transform([features])
        prediction      = model.predict(features_scaled)[0]
        probabilities   = model.predict_proba(features_scaled)[0]
        confidence      = probabilities[prediction] * 100
        label = "Deepfake Voice" if prediction == 1 else "Real Voice"

        print()
        print("=" * 40)
        print(f"  Result     : {label}")
        print(f"  Confidence : {confidence:.1f}%")
        print(f"  Real probability  : {probabilities[0]*100:.1f}%")
        print(f"  Fake probability  : {probabilities[1]*100:.1f}%")
        print("=" * 40)

    os.remove(test_filename)

---
## Step 11 — Run Streamlit Web App via ngrok

This launches the full Streamlit web interface inside Colab.
We use **ngrok** instead of localtunnel — no password page, no IP needed.

### Before running this step:
1. Go to **[https://ngrok.com](https://ngrok.com)** and create a free account
2. After signing in, go to: **Dashboard --> Your Authtoken**
3. Copy your token — it looks like: `2abc123XYZ_abcdefghijklmnop`
4. Paste it into **Step 11d** below where it says `YOUR_NGROK_TOKEN`

> The ngrok URL is temporary. It stops working when this Colab session ends.
> Every new session gives you a new URL.

In [ ]:
# Step 11a — Install streamlit and pyngrok

!pip install streamlit pyngrok --quiet

print("streamlit and pyngrok installed!")

In [ ]:
# Step 11b — Upload your streamlit_app.py
#
# Upload the streamlit_app.py from your project folder on your computer.

print("Upload your streamlit_app.py file...")
uploaded = files.upload()

if 'streamlit_app.py' in uploaded:
    print("streamlit_app.py uploaded successfully!")
    print("You can now continue to Step 11c.")
else:
    print("streamlit_app.py not found in the upload. Please try again.")


In [ ]:
# Step 11c — Authenticate ngrok
#
# How to get your token:
# 1. Go to https://ngrok.com and sign up free
# 2. After login: Dashboard --> Getting Started --> Your Authtoken
# 3. Click the copy button next to your full token
# 4. Paste it below replacing YOUR_NGROK_TOKEN
#
# Your token is very long — something like:
#   2abc123XYZ_abcdefghijklmnopqrstuvwxyz1234567890ABCDEF

NGROK_TOKEN = "YOUR_NGROK_TOKEN"   # <-- paste your full token here

import subprocess

# Authenticate using the ngrok CLI directly (more reliable than pyngrok)
result = subprocess.run(
    ["ngrok", "authtoken", NGROK_TOKEN],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("ngrok authenticated successfully!")
    print("You can now run Step 11d.")
else:
    print("Authentication failed:")
    print(result.stderr)
    print()
    print("Make sure you copied the FULL token from:")
    print("https://dashboard.ngrok.com/get-started/your-authtoken")


In [ ]:
# Step 11d — Launch Streamlit + ngrok
#
# Starts Streamlit on port 8501 and opens an ngrok tunnel.
# After running, click the URL printed below — no password needed.
# Keep this cell running while using the app.
# Click the stop button to shut down.

import subprocess
import threading
import time
import re

# Kill any leftover processes from a previous run
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "ngrok"],     capture_output=True)
time.sleep(2)

# Step 1 — Start Streamlit in the background
print("Starting Streamlit server...")
streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "streamlit_app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)
print("Streamlit is running on port 8501")

# Step 2 — Start ngrok tunnel in background and capture the URL
print("Opening ngrok tunnel...")

ngrok_output = []

def run_ngrok():
    proc = subprocess.Popen(
        ["ngrok", "http", "8501", "--log=stdout"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        ngrok_output.append(line)

ngrok_thread = threading.Thread(target=run_ngrok, daemon=True)
ngrok_thread.start()
time.sleep(4)

# Step 3 — Get the public URL from the ngrok API
import urllib.request, json as _json

try:
    with urllib.request.urlopen("http://localhost:4040/api/tunnels") as resp:
        tunnels = _json.loads(resp.read())
    public_url = tunnels["tunnels"][0]["public_url"]
    print()
    print("=" * 55)
    print("  Your Streamlit app is LIVE!")
    print(f"  URL --> {public_url}")
    print("=" * 55)
    print()
    print("  1. Click the URL above")
    print("  2. App opens immediately — no password needed")
    print()
    print("  Keep this cell running while using the app.")
    print("  Run Step 11e or click stop to shut down.")
except Exception as e:
    print()
    print("Could not get URL automatically. Check the logs:")
    for line in ngrok_output[-10:]:
        print(line, end="")
    print()
    print("Make sure Step 11c was run successfully first.")


In [ ]:
# Step 11e — Stop the app (run this when you are done)
#
# Run this cell to cleanly shut down Streamlit and close the ngrok tunnel.

from pyngrok import ngrok

ngrok.kill()

try:
    streamlit_process.terminate()
    print("Streamlit stopped.")
except:
    print("Streamlit process already stopped.")

print("ngrok tunnel closed.")
print("App is now offline.")

---

## Summary

**Required steps:**

1. Install libraries
2. Import packages
3. Upload dataset
4. Validate dataset
5. Feature extraction
6. Train Random Forest
7. Evaluate model
8. Preview audio visuals *(optional)*
9. Save and download model
10. Test single prediction *(optional)*
11. Run Streamlit app via ngrok

---

### Step 11 Quick Reference

1. **11a** — Install streamlit and pyngrok
2. **11b** — Upload your `streamlit_app.py` from your computer
3. **11c** — Paste your ngrok token from dashboard.ngrok.com
4. **11d** — Launch the app — click the URL in the output
5. **11e** — Run this to shut down the app when finished

---
*Deepfake Audio Detection System  |  Vishing Defense Project  |  Python + librosa + scikit-learn + Streamlit*
